In [1]:
import os
import json
from typing import List, Optional
import asyncio
import warnings
import numpy as np
warnings.filterwarnings('ignore')

In [2]:
# Core LlamaIndex imports
from llama_index.core import (
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    Document,
    Settings,
    DocumentSummaryIndex,
    KeywordTableIndex
)

In [3]:
from llama_index.core.retrievers import (
    BaseRetriever,
    VectorIndexRetriever,
    AutoMergingRetriever,
    RecursiveRetriever,
    QueryFusionRetriever
)

In [4]:
from llama_index.core.indices.document_summary import (
    DocumentSummaryIndexLLMRetriever,
    DocumentSummaryIndexEmbeddingRetriever,
)

In [5]:
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.embeddings import BaseEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


In [6]:
# Advanced retriever imports
from llama_index.retrievers.bm25 import BM25Retriever

In [7]:
# Sentence transformers
from sentence_transformers import SentenceTransformer

In [8]:
from langchain_ollama import ChatOllama,OllamaEmbeddings

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [37]:
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings

Settings.embed_model = OllamaEmbedding(
    model_name="mxbai-embed-large:latest",
    base_url="http://localhost:11434",
)

In [10]:
# Sample data for the lab - AI/ML focused documents
SAMPLE_DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
    "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.",
    "Supervised learning uses labeled training data to learn a mapping from inputs to outputs.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative AI can create new content including text, images, code, and more.",
    "Large language models are trained on vast amounts of text data to understand and generate human-like text."
]

# Consistent query examples used throughout the lab
DEMO_QUERIES = {
    "basic": "What is machine learning?",
    "technical": "neural networks deep learning", 
    "learning_types": "different types of learning",
    "advanced": "How do neural networks work in deep learning?",
    "applications": "What are the applications of AI?",
    "comprehensive": "What are the main approaches to machine learning?",
    "specific": "supervised learning techniques"
}

In [29]:
documents=[Document(text=text) for text in SAMPLE_DOCUMENTS]

In [30]:
from pprint import pprint
pprint(documents[0].text)

('Machine learning is a subset of artificial intelligence that focuses on '
 'algorithms that can learn from data.')


In [64]:
nodes=SentenceSplitter().get_nodes_from_documents(documents)

In [34]:
for i in nodes:
    print(i.text)

Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
Natural language processing enables computers to understand, interpret, and generate human language.
Computer vision allows machines to interpret and understand visual information from the world.
Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.
Supervised learning uses labeled training data to learn a mapping from inputs to outputs.
Unsupervised learning finds hidden patterns in data without labeled examples.
Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.
Generative AI can create new content including text, images, code, and more.
Large language models are trained on vast amounts of text data to understand and generate human-like text.


In [38]:
# creating various indexes
vector_index=VectorStoreIndex.from_documents(documents)

In [44]:
from llama_index.llms.ollama import Ollama
Settings.llm = Ollama(
    model="minimax-m3:cloud",
    request_timeout=300,
)

In [45]:
document_summary_index=DocumentSummaryIndex.from_documents(documents)

current doc id: 0dcdca40-4694-41d3-882d-d2f34a9ddf5f
current doc id: ddd47f68-4fa7-450f-954b-c7c55a91c2b8
current doc id: 1b6078e9-8cbc-4cc2-b103-1b25a7696366
current doc id: 2ee5c479-d107-4185-943f-8d74b4ab2d8e
current doc id: ed73f75f-8ba6-4b71-aff4-e1945138a5f0
current doc id: 3bb6e0da-3359-424a-8d67-f035ec6e0ee8
current doc id: 9eafadb3-0236-441a-9a37-f00181c08115
current doc id: ca543e67-2ba6-4a1a-9e61-fc5a65e64447
current doc id: fb012022-f6a7-433b-a45d-0236a84df616
current doc id: 0ea3d669-d90a-4da1-9168-cbfae4841c91


In [46]:
keyword_index=KeywordTableIndex.from_documents(documents)

## 1. Vector Index Retriever - The Foundation

The Vector Index Retriever uses vector embeddings to find semantically related content, making it ideal for general-purpose search and widely used in retrieval-augmented generation (RAG) pipelines.

**How it works**: 
- Documents are split into nodes and embedded using the configured embedding model
- Query is converted to an embedding vector
- Returns nodes ranked by cosine similarity to the query embedding
- Generates embeddings in batches of 2048 nodes by default

**When to use:**
- General-purpose semantic search (most common use case)
- Finding conceptually related content based on meaning rather than exact keywords
- RAG pipelines where semantic understanding is crucial
- When exact keyword matching isn't the primary requirement

**Key characteristics from authoritative source:**
- **Stores embeddings for each document chunk** (VectorStoreIndex foundation)
- **Best for semantic retrieval** based on meaning and context
- **Commonly used in LLM pipelines** for retrieval-augmented generation

**Strengths**: 
- Excellent semantic understanding and context awareness
- Handles synonyms and related concepts effectively
- Works well with natural language queries

**Limitations**: 
- May miss exact keyword matches when specific terms are crucial
- Requires a good embedding model for optimal performance
- Can be computationally intensive for large document collections


In [47]:
# Basic vector retriever
vector_retriever=VectorIndexRetriever(
    index=vector_index,
    similarity_top_k=3
)

In [49]:
query=DEMO_QUERIES["basic"]
query

'What is machine learning?'

In [62]:
node=vector_retriever.retrieve(query)

In [63]:
for i,node in enumerate(node,1):
    print(f"{i}. Score:{node.score:.4f}")
    print(f"  Text: {node.text}")
    print('='*200)

1. Score:0.8630
  Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
2. Score:0.7039
  Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.
3. Score:0.6810
  Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs.


## 2. BM25 Retriever - Advanced Keyword-Based Search

BM25 is a keyword-based retrieval method that improves on TF-IDF by addressing some of its key limitations. It's widely used in production search systems including Elasticsearch and Apache Lucene.

### Understanding TF-IDF: The Foundation

Before diving into BM25, let's understand **TF-IDF** (Term Frequency-Inverse Document Frequency), which BM25 builds upon:

**Term Frequency (TF)**: Measures how often a word appears in a document
- Example: If "neural" appears 3 times in a 100-word document, TF = 3/100 = 0.03

**Inverse Document Frequency (IDF)**: Measures how rare a word is across all documents
- Example: If "neural" appears in only 2 out of 1000 documents, IDF = log(1000/2) = 6.21
- Common words like "the" have low IDF; rare technical terms have high IDF

**TF-IDF Score**: TF × IDF
- Highlights words that are frequent in one document but rare across the collection
- Developed by Karen Spärck Jones, who pioneered the concept of term specificity

### How BM25 Improves Upon TF-IDF

**Key BM25 Improvements:**

1. **Term Frequency Saturation**: BM25 reduces the impact of repeated terms using term frequency saturation
   - Problem: In TF-IDF, if a word appears 100 times vs 10 times, the score increases linearly
   - Solution: BM25 uses a saturation function that plateaus after a certain frequency

2. **Document Length Normalization**: BM25 adjusts for document length, making it more effective for keyword-based search
   - Problem: In TF-IDF, longer documents have unfair advantages
   - Solution: BM25 normalizes scores based on document length relative to average

3. **Tunable Parameters**: Allows fine-tuning for different types of content
   - k1 ≈ 1.2: Controls term frequency saturation (how quickly scores plateau)
   - b ≈ 0.75: Controls document length normalization (0=none, 1=full)

### When to Use BM25

**Ideal for:**
- Technical documentation where exact terms matter
- Legal documents with specific terminology
- Product catalogs with precise specifications
- Academic papers with specialized vocabulary
- Applications requiring keyword-based retrieval rather than semantic similarity

**Advantages:**
- Excellent precision for exact term matches
- Fast computational performance
- Proven effectiveness in production systems
- No training required (unlike neural approaches)
- Interpretable scoring mechanism

**Limitations:**
- No semantic understanding (doesn't handle synonyms)
- Struggles with typos and variations
- Limited context understanding
- Requires careful parameter tuning for optimal performance


In [59]:
import Stemmer

In [65]:
# create BM25 retriever with default parameters
bm25_retriever=BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=3,
    stemmer=Stemmer.Stemmer("english"),
    language="english"
)

In [66]:
query=DEMO_QUERIES['technical']
query

'neural networks deep learning'

In [67]:
node_bm25=bm25_retriever.retrieve(query)

In [73]:
for i,node in enumerate(node_bm25,1):
    print(f"{i}. Score:{node.score:.4f}")
    print(f"  Text: {node.text}")
    
     # Demonstrate TF-IDF concept manually
    text_lower = node.text.lower()
    query_terms = query.lower().split()
    found_terms = [term for term in query_terms if term in text_lower]
        
    if found_terms:
        print(f"   → BM25 would boost this result for terms: {found_terms}")
    print('='*200)
    
print("BM25 Concept Demonstration:")
print("1. TF-IDF Foundation:")
print("   - Term Frequency: How often words appear in document")
print("   - Inverse Document Frequency: How rare words are across collection")
print("   - TF-IDF = TF × IDF (balances frequency vs rarity)")
print()
print("2. BM25 Improvements:")
print("   - Saturation: Prevents over-scoring repeated terms")
print("   - Length normalization: Prevents long document bias")
print("   - Tunable parameters: k1 (saturation) and b (length adjustment)")
print()
print("3. Real-world Usage:")
print("   - Elasticsearch default scoring function")
print("   - Apache Lucene/Solr standard")
print("   - Used in 83% of text-based recommender systems")
print("   - Developed by Robertson & Spärck Jones at City University London")

1. Score:2.5203
  Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
   → BM25 would boost this result for terms: ['neural', 'networks', 'deep', 'learning']
2. Score:0.3372
  Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.
   → BM25 would boost this result for terms: ['learning']
3. Score:0.3024
  Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
   → BM25 would boost this result for terms: ['learning']
BM25 Concept Demonstration:
1. TF-IDF Foundation:
   - Term Frequency: How often words appear in document
   - Inverse Document Frequency: How rare words are across collection
   - TF-IDF = TF × IDF (balances frequency vs rarity)

2. BM25 Improvements:
   - Saturation: Prevents over-scoring repeated terms
   - Length normalization: Prevents long document bias
   - Tunable parameters: k